# Desctription

This notebooks will be used to generate the formatted data requiorte dforr the inference of pxi2pix pretraiend Generator so that we can use that style transferred data for training the segformer which we then believe would help in better convergence and ultimately increase the dice score from 69% to 75%. 
Lets do it 

In [24]:
import pandas as pd 
import os 
import cv2 
import numpy as np 
import matplotlib.pyplot as plt 
from PIL import Image

In [25]:
def normalize_image(image):
    image = image.astype(np.float32)
    image = image - np.min(image) / (np.max(image) - np.min(image))
    return image

In [26]:


# we need to create a folder that has train A, train B, test A, test B
kwave_simulated_data_path = "/home/user/data/phyusformer_data/all_combined_synthetic_datasets_V3_BM/scans"
# images_path = [os.path.join(kwave_simulated_data_path, "scans")]
images_path = [os.path.join(kwave_simulated_data_path, x) for x in os.listdir(kwave_simulated_data_path) if x.endswith(".pkl")]
# target save directory 
save_path = "/home/user/data/phyusformer_data/post_miccai_exps/data/pix2pix_large_inference_data"
os.makedirs(save_path, exist_ok=True)
# we need to create a folder that has testA and testB
test_A_path = os.path.join(save_path, "testA")
test_B_path = os.path.join(save_path, "testB")
os.makedirs(test_A_path, exist_ok=True)
os.makedirs(test_B_path, exist_ok=True)

# assert that files are present in simulated folder path
assert os.path.exists(images_path[0]), "Files are not present in the simulated folder path"

Lets load all the scan files and load the image in the testA while for the same image name save the blank image in testB as it does not matter and we are doing it for the reason of fuck all i know the creators were fucking super high while writing this repo but i gotta respect that as it saves the time but a bit of document or comment would have been super nice, init eh?

In [27]:
for file in sorted(images_path):
    # print(file)
    # load the pkl file 
    data = pd.read_pickle(file)['noisy_us_scan_harmonic']
    id = file.split("/")[-1].split(".")[0]
    data = normalize_image(data)
    data = data.astype(np.uint8)
    image = Image.fromarray(data)
    image.save(os.path.join(test_A_path, f"{id}.png"))
    # save the image in testB
    image = Image.fromarray(np.zeros_like(data))
    image.save(os.path.join(test_B_path, f"{id}.png"))
    break

Lets create a folder "Test"  which has aligned testA and testB 

In [34]:
# Creating the Aligned Dataset for the Pix2Pix 
import os

from PIL import Image


def get_file_paths(folder):
    image_file_paths = []
    for root, dirs, filenames in os.walk(folder):
        filenames = sorted(filenames)
        for filename in filenames:
            input_path = os.path.abspath(root)
            file_path = os.path.join(input_path, filename)
            if filename.endswith('.png') or filename.endswith('.jpg'):
                image_file_paths.append(file_path)

        break  # prevent descending into subfolders
    return image_file_paths


def align_images(a_file_paths, b_file_paths, target_path):
    if not os.path.exists(target_path):
        os.makedirs(target_path)

    for i in range(len(a_file_paths)):
        img_a = Image.open(a_file_paths[i])
        img_b = Image.open(b_file_paths[i])
        assert(img_a.size == img_b.size)

        aligned_image = Image.new("RGB", (img_a.size[0] * 2, img_a.size[1]))
        aligned_image.paste(img_a, (0, 0))
        aligned_image.paste(img_b, (img_a.size[0], 0))
        id = a_file_paths[i].split("/")[-1].split(".")[0]
        print(id)
        aligned_image.save(os.path.join(target_path, f"{id}.png"))
        # break

In [35]:

# parser.add_argument(
#     '--dataset-path',
#     dest='dataset_path',
#     help='Which folder to process (it should have subfolders testA, testB, trainA and trainB'
# )
# args = parser.parse_args()

# dataset_folder = args.dataset_path
dataset_folder = "/home/user/data/phyusformer_data/post_miccai_exps/data/pix2pix_large_inference_data"
os.makedirs(dataset_folder, exist_ok=True)
# print(dataset_folder)

test_a_path = os.path.join(dataset_folder, 'testA')
test_b_path = os.path.join(dataset_folder, 'testB')
test_a_file_paths = get_file_paths(test_a_path)
test_b_file_paths = get_file_paths(test_b_path)
assert(len(test_a_file_paths) == len(test_b_file_paths))
test_path = os.path.join(dataset_folder, 'test')
align_images(test_a_file_paths, test_b_file_paths, test_path)

scan_1
scan_10
scan_100
scan_1000
scan_1001
scan_1002
scan_1003
scan_1004
scan_1005
scan_1006
scan_1007
scan_1008
scan_1009
scan_101
scan_1010
scan_1011
scan_1012
scan_1013
scan_1014
scan_1015
scan_1016
scan_1017
scan_1018
scan_1019
scan_102
scan_1020
scan_1021
scan_1022
scan_1023
scan_1024
scan_1025
scan_1026
scan_1027
scan_1028
scan_1029
scan_103
scan_1030
scan_1031
scan_1032
scan_1033
scan_1034
scan_1035
scan_1036
scan_1037
scan_1038
scan_1039
scan_104
scan_1040
scan_1041
scan_1042
scan_1043
scan_1044
scan_1045
scan_1046
scan_1047
scan_1048
scan_1049
scan_105
scan_1050
scan_1051
scan_1052
scan_1053
scan_1054
scan_1055
scan_1056
scan_1057
scan_1058
scan_1059
scan_106
scan_1060
scan_1061
scan_1062
scan_1063
scan_1064
scan_1065
scan_1066
scan_1067
scan_1068
scan_1069
scan_107
scan_1070
scan_1071
scan_1072
scan_1073
scan_1074
scan_1075
scan_1076
scan_1077
scan_1078
scan_1079
scan_108
scan_1080
scan_1081
scan_1082
scan_1083
scan_1084
scan_1085
scan_1086
scan_1087
scan_1088
scan_1089
scan

After it, we run the following command to get the inference from the pretrained Pix2Pix 
"python test.py --dataroot /home/user/data/phyusformer_data/post_miccai_exps/data/pix2pix_large_inference_data/ --name pix2pix_source2target_highres --model pix2pix --use_wandb --input_nc 1 --output_nc 1 --wandb_project_name pix2pix_source2target_highres --gpu_id 0 --checkpoints_dir /home/user/data/phyusformer_data/post_miccai_exps/pix2pix_checkpoints/ --direction AtoB --results_dir results/pix2pix_source2target_highres_large_data_inference --num_test 10000"


Now lets put the masks with the same strategy but this time in the resutls folder so that we can create the index html as well as then download the data to train the segformer.

In [47]:
masks_path

['/home/user/data/phyusformer_data/all_combined_synthetic_datasets_V3_BM/labels/label_2544.pkl',
 '/home/user/data/phyusformer_data/all_combined_synthetic_datasets_V3_BM/labels/label_5006.pkl',
 '/home/user/data/phyusformer_data/all_combined_synthetic_datasets_V3_BM/labels/label_5292.pkl',
 '/home/user/data/phyusformer_data/all_combined_synthetic_datasets_V3_BM/labels/label_3197.pkl',
 '/home/user/data/phyusformer_data/all_combined_synthetic_datasets_V3_BM/labels/label_3127.pkl',
 '/home/user/data/phyusformer_data/all_combined_synthetic_datasets_V3_BM/labels/label_7401.pkl',
 '/home/user/data/phyusformer_data/all_combined_synthetic_datasets_V3_BM/labels/label_1496.pkl',
 '/home/user/data/phyusformer_data/all_combined_synthetic_datasets_V3_BM/labels/label_4098.pkl',
 '/home/user/data/phyusformer_data/all_combined_synthetic_datasets_V3_BM/labels/label_5364.pkl',
 '/home/user/data/phyusformer_data/all_combined_synthetic_datasets_V3_BM/labels/label_367.pkl',
 '/home/user/data/phyusformer_d

In [52]:
kwave_simulated_data_path = "/home/user/data/phyusformer_data/all_combined_synthetic_datasets_V3_BM/labels"
# images_path = [os.path.join(kwave_simulated_data_path, "scans")]
masks_path = [os.path.join(kwave_simulated_data_path, x) for x in os.listdir(kwave_simulated_data_path) if x.endswith(".pkl")]
print(len(masks_path))
save_target = "/home/user/haris/pytorch-CycleGAN-and-pix2pix/results/pix2pix_source2target_highres_large_data_inference/pix2pix_source2target_highres/test_latest/images"
for file in sorted(masks_path):
    id = file.split("/")[-1].split(".")[0].replace("label_", "scan_")
    mask_data = pd.read_pickle(file)['clean_phantom_binary']
    mask_data = (mask_data - np.min(mask_data)) / (np.max(mask_data) - np.min(mask_data))
    mask_data = (mask_data  > 0.5).astype(np.uint8)
    mask_data = mask_data * 255
    mask_data = mask_data.astype(np.uint8)
    # assert np.max(mask_data) == 1
    # assert np.min(mask_data) == 0
    image = Image.fromarray(mask_data)
    image.save(os.path.join(save_target, f"{id}_mask.png"))
    break

9243


Once inference is done, download this data for trainign segformer 
we can alos do the error analysis for this data and run the error quality metrics -> to be done later jaan


Its going on but feels its super difficult for AI to understand the inferred data from pix2pix as meaninful because the mask does not feel aligned to that shit 

Anyways, plan is to train a big CYCLEGAN using US30k as the target domain and bezier+highreskwave data as source domain and the ratio is stil 10k images to 30k iamges but lets see how it turns out. 
